_Date: 2026-08-14_


# Chirp Pulse on SHFQC SG Channel 1

Plays a single linear-FM **chirp pulse** -- instantaneous frequency sweeping linearly
across a 100 MHz span centered at 6 GHz -- out of SG channel 1 (front panel) of an SHFQC,
repeated in a loop so it can be observed live on a scope or spectrum analyzer.

Raw LabOneQ DSL only (no `laboneq_applications`, no qubit/QPU abstraction, no QA/readout
channel) -- this is pure signal generation, not a qubit experiment.

**Centering approach**: the SG channel's LO is set to 6.0 GHz and its digital oscillator
(IF) to 0 Hz. The chirp pulse's own envelope then sweeps +/-50 MHz around baseband, which
becomes the +/-50 MHz RF sweep around the 6 GHz LO -- no separate frequency-sweep
parameter is needed.


In [ ]:
import numpy as np
from laboneq.simple import *

## 1. Device Setup & Calibration -- SHFQC SG channel 1

**Fill in before running on real hardware:**
- `address` -- your SHFQC's device serial (e.g. `dev12345`, check the LabOne Web UI or the
  label on the front panel).
- `device_options` -- your SHFQC's installed options string (LabOne Web UI -> Device tab,
  or `*OPT?`).
- `host` -- the LabOne dataserver host (`"localhost"` if running on the same PC as the
  dataserver).
- `range` in the `SignalCalibration` below -- a safe output range (dBm) for your SG
  channel/setup.

`do_emulation=True` is kept as the default below so nothing is sent to the instrument
until you've checked the values above -- flip to `False` once confirmed.


In [ ]:
CENTER_FREQ = 6.0e9   # chirp center frequency (Hz) -- set via the SG channel's LO

device_setup = DeviceSetup(uid="shfqc_chirp_setup")
device_setup.add_dataserver(host="10.42.193.4", port="8004")

shfqc = SHFQC(
    uid="device_shfqc",
    address="dev12073",
    interface="1GbE",
    device_options="SHFQC/LRT/PLUS/QC6CH/RTR",
    reference_clock_source="internal",
)
device_setup.add_instruments(shfqc)

device_setup.add_connections(
    "device_shfqc",
    create_connection(to_signal="chirp/drive", ports="SGCHANNELS/0/OUTPUT", type="iq"),
)

calibration = Calibration()
calibration["chirp/drive"] = SignalCalibration(
    oscillator=Oscillator(uid="chirp_if_osc", frequency=0.0, modulation_type=ModulationType.HARDWARE),
    local_oscillator=Oscillator(uid="chirp_lo", frequency=CENTER_FREQ),
    range=-10,   # TODO: set a safe output range (dBm) for your setup
)
device_setup.set_calibration(calibration)


## 2. Chirp Pulse Definition

A linear FM chirp needs its instantaneous frequency to ramp from `-span/2` to `+span/2`
across the pulse. Using `register_pulse_functional`'s `x` in `[-1, 1]` normalized-envelope
coordinate (real time `t = x * length/2`), integrating `f(t) = (span/2) * x` over the
envelope gives:

```
phase(x)    = (pi/4) * length * span * (x**2 - 1)
envelope(x) = exp(i * phase(x))
```

(`phase(-1) == phase(+1) == 0` -- consistent with a symmetric sweep whose average
frequency is 0, so the net phase accumulated over the whole pulse is zero.)


In [ ]:
SWEEP_SPAN = 100e6     # total chirp span (Hz), centered at CENTER_FREQ -> +/-50 MHz
PULSE_LENGTH = 1e-6    # chirp duration (s)
AMPLITUDE = 0.5        # pulse amplitude (0-1, fraction of full output scale)


@pulse_library.register_pulse_functional
def chirp_pulse(x, length, sweep_span, **_):
    phase = (np.pi / 4) * length * sweep_span * (x**2 - 1)
    return np.exp(1j * phase)


chirp = chirp_pulse(uid="chirp", length=PULSE_LENGTH, sweep_span=SWEEP_SPAN, amplitude=AMPLITUDE)


## 3. Experiment

Single "drive" signal, no acquire. The chirp is played repeatedly (`REPEAT_COUNT` shots)
with a relax gap between repeats, so it can be captured continuously on an external
instrument.

`marker1` (the only marker key LabOne Q exposes on SHFSG/SHFQC SG channels) is enabled
alongside the pulse, riding out on that SG channel's front-panel marker/trigger output --
high for the full chirp duration, so a scope can trigger on it and capture in sync with
each chirp.


In [ ]:
REPEAT_COUNT = 1000
RELAX_TIME = 10e-6

exp = Experiment(uid="chirp_experiment", signals=[ExperimentSignal("drive")])
exp.set_signal_map({"drive": "chirp/drive"})

with exp.acquire_loop_rt(uid="shots", count=REPEAT_COUNT, averaging_mode=AveragingMode.CYCLIC):
    with exp.section(uid="chirp_section"):
        exp.play(signal="drive", pulse=chirp, marker={"marker1": {"enable": True}})
    with exp.section(uid="relax", play_after="chirp_section"):
        exp.delay(signal="drive", time=RELAX_TIME)


## 4. Compile & Run

The cell below reruns `session.run()` in a loop so the burst repeats continuously --
arm/trigger the scope on the marker output, then interrupt the kernel (or the Jupyter stop
button) once you've captured what you need.


In [ ]:
session = Session(device_setup)
session.connect(do_emulation=False)   # set False once address/device_options/range above are confirmed

compiled_exp = session.compile(exp)
session.run(compiled_exp)

# try:
#     while True:
#         session.run(compiled_exp)
# except KeyboardInterrupt:
#     print("Stopped.")


## 5. Pulse Sheet Viewer

Same pattern as the chevron notebook: `show_pulse_sheet` writes a self-contained HTML
file (a ~1.4 MB inlined JS bundle), which VS Code's notebook link/preview can't execute
properly -- force-open it in the system browser instead.


In [ ]:
import glob
import os
import webbrowser

show_pulse_sheet("chirp_pulse_sheet", compiled_exp, max_events_to_publish=2000)

latest_pulse_sheet = max(glob.glob("chirp_pulse_sheet_*.html"), key=os.path.getmtime)
webbrowser.open(f"file://{os.path.abspath(latest_pulse_sheet)}")
print(f"Opened {latest_pulse_sheet} in your default browser")


## 6. Instantaneous-Frequency Sanity Plot

Quick analytic check, independent of compiling/running anything: instantaneous frequency
`f(t) = SWEEP_SPAN * t / PULSE_LENGTH` should sweep linearly from -50 MHz to +50 MHz
across the pulse -- i.e. 5.95-6.05 GHz once centered at `CENTER_FREQ`.


In [ ]:
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "browser"

t = np.linspace(-PULSE_LENGTH / 2, PULSE_LENGTH / 2, 501)
inst_freq_hz = SWEEP_SPAN * t / PULSE_LENGTH

fig = go.Figure()
fig.add_trace(go.Scatter(x=t * 1e6, y=(CENTER_FREQ + inst_freq_hz) / 1e9, line=dict(color="darkorange", width=2)))
fig.update_xaxes(title_text="Time (us)")
fig.update_yaxes(title_text="Instantaneous frequency (GHz)")
fig.update_layout(title="Chirp instantaneous frequency vs. time", template="plotly_white",
                   paper_bgcolor="white", plot_bgcolor="white", width=800, height=500, autosize=False)
fig.show()
